In [ ]:
import os.path

from scipy.stats import entropy
from sklearn.ensemble import RandomForestClassifier
from dotenv import load_dotenv
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
from sklearn.model_selection import StratifiedKFold, cross_val_score


In [ ]:
import matplotlib.style as _ms

# Patch for mplcyberpunk expecting matplotlib.style.core
if not hasattr(_ms, "core"):
    _ms.core = _ms

import mplcyberpunk as mpl

In [ ]:
plt.style.use('cyberpunk')
mpl.add_glow_effects()

In [ ]:
load_dotenv()

In [ ]:
# # Download latest version
#
# path = kagglehub.competition_download('playground-series-s6e9',
#                                       output_dir='data/ev_purchase',
#                                       )
#
# print("Path to competition files:", path)

In [ ]:
test = pd.read_csv('data/ev_purchase/test.csv', index_col='id')
train = pd.read_csv('data/ev_purchase/train.csv', index_col='id')

In [ ]:
train.describe()

In [ ]:
train.head(10)

In [ ]:
cat_columns = train.select_dtypes(include=str).columns
# we need check unique values in categorical columns to see if there are any inconsistencies

train[cat_columns].nunique()

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['Age',
                                   'Annual_Income_USD',
                                   'Daily_Commute_km',
                                   'Number_of_Cars_Owned',
                                   'Charging_Stations_Near_Home',
                                   'Charging_Stations_Near_Work',
                                   'Environmental_Concern_Level'
                                   ]),
        ('label', OrdinalEncoder(), ['Gender',
                                     'Home_Charging_Possible',
                                     'Subsidy_Available',
                                     'Range_Anxiety_Level']),
        ('cat', OneHotEncoder(handle_unknown='ignore'), ['City_Type',
                                                         'Current_Car_Type']),
    ]
)

In [ ]:
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100,
                                          random_state=42,
                                          n_jobs=-1))
])

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    train.drop(columns=['Will_Buy_EV']),
    train['Will_Buy_EV'],
    test_size=0.2,
    random_state=42
)

In [ ]:
# save trained pipeline to disk
import joblib

model_name = 'ev_purchase_pipeline.pkl'

if os.path.exists(model_name):
    print(f'Model file {model_name} already exists. Loading the existing model.')
    pipeline = joblib.load(model_name)
else:
    print(f'Model file {model_name} does not exist. Creating a new model.')
    pipeline.fit(X_train, y_train)
    joblib.dump(pipeline, model_name)
    print("Pipeline saved to 'ev_purchase_pipeline.pkl'")

In [ ]:
y_pred_pipeline = pipeline.predict(X_val)
accuracy_pipeline = accuracy_score(y_val, y_pred_pipeline)
print(f"Validation Accuracy (Pipeline): {accuracy_pipeline:.4f}")

In [ ]:
conf_matrix = pd.crosstab(y_val, y_pred_pipeline, rownames=['Actual'], colnames=['Predicted'])
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues')
plt.title(f'Confusion Matrix (Validation Set) - Accuracy: {accuracy_pipeline:.4f}')
plt.show()

In [ ]:
clf_report = classification_report(y_val, y_pred_pipeline)
print(clf_report)

In [ ]:
# Feature importance from the Random Forest model
feature_importances = pipeline.named_steps['classifier'].feature_importances_
# Get feature names from the preprocessor
num_features = ['Age', 'Annual_Income_USD', 'Daily_Commute_km', 'Number_of_Cars_Owned', 'Charging_Stations_Near_Home', 'Charging_Stations_Near_Work', 'Environmental_Concern_Level']
label_features = ['Gender', 'Home_Charging_Possible', 'Subsidy_Available', 'Range_Anxiety_Level']
cat_features = pipeline.named_steps['preprocessor'].named_transformers_['cat'].get_feature_names_out(['City_Type', 'Current_Car_Type']).tolist()
all_features = num_features + label_features + cat_features
# Create a DataFrame for feature importances
feature_importance_df = pd.DataFrame({
    'Feature': all_features,
    'Importance': feature_importances
}).sort_values(by='Importance', ascending=False)

In [ ]:
# Plot feature importances
plt.figure(figsize=(12, 8))
sns.barplot(x='Importance', y='Feature', data=feature_importance_df)
plt.title('Feature Importances from Random Forest Classifier')
plt.show()

In [ ]:
# let's prepare the data
# 1. use small piece of the training data ( 1000 rows ) to speed up the hyperparameter optimization process
X_train_small, _, y_train_small, _ = train_test_split(
    X_train,
    y_train,
    train_size=1000,
    random_state=42
)

# 2. Transform the small training data using the preprocessor
X_train_small_transformed = preprocessor.fit_transform(X_train_small)

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
# use Optuna to optimize hyperparameters of the Random Forest model
# {'n_estimators': 425,
#  'max_depth': 22,
#  'min_samples_split': 7,
#  'min_samples_leaf': 4}

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 10, 30),
        'min_samples_split': trial.suggest_int('min_samples_split', 5, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        # 'max_features': trial.suggest_categorical('max_features', ['auto', 'sqrt', 'log2']),
        # 'criterion': trial.suggest_categorical('criterion', ['gini', 'entropy']),
    }

    model = RandomForestClassifier(**params,
                                   criterion='entropy',
                                   random_state=42)
    cv_scores = cross_val_score(model,
                                X_train_small_transformed,
                                y_train_small,
                                cv=cv,
                                scoring='accuracy')
    accuracy = cv_scores.mean()

    if trial.should_prune():
        raise optuna.exceptions.TrialPruned()

    return accuracy


In [ ]:
# pruner = optuna.pruners.PatientPruner(optuna.pruners.SuccessiveHalvingPruner(), patience=10)

pruner = optuna.pruners.MedianPruner()

study = optuna.create_study(direction='maximize',
                            sampler=optuna.samplers.TPESampler(seed=42),
                            pruner=pruner)

study.optimize(objective,
               n_trials=50,
               n_jobs=-1,
               show_progress_bar=True)

print("Best hyperparameters: ", study.best_params)


In [ ]:
# Visualize the optimization history
optuna.visualization.plot_optimization_history(study)

In [ ]:
optuna.visualization.plot_param_importances(study)

In [ ]:
optuna.visualization.plot_parallel_coordinate(study)

In [ ]:
best_params = study.best_params
best_params

In [ ]:
score = study.best_value
print(f'Best cross-validated accuracy: {score:.4f}')

In [ ]:
# Train the final model with the best hyperparameters
best_params = study.best_params
final_clf = RandomForestClassifier(
    **best_params,
    criterion='entropy',
    random_state=42,
    n_jobs=-1
)

final_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', final_clf)
])

final_pipeline.fit(X_train, y_train)
y_pred_final = final_pipeline.predict(X_val)
final_accuracy = accuracy_score(y_val, y_pred_final)
print(f'Final Model Accuracy: {final_accuracy:.4f}')